# EvolveAI Brain v3 — Semantic Memory Build\n\nBuild persistent, source-grounded memory from Brain v2 episodic records. This notebook does not train the SNN. Concepts are heuristic candidates; graph edges indicate co-occurrence, not verified facts.

In [ ]:
from pathlib import Path\nimport subprocess\nREPO=Path('/content/EvolveAI'); BRANCH='evolveai-maxgpu-one-shot'\nif not REPO.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/swasthik-s/EvolveAI.git',str(REPO)],check=True)\nelse:\n    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)\n    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)\n    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)\n%cd /content/EvolveAI/snn\n

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\nBRAIN_V2=Path('/content/drive/MyDrive/EvolveAI/brain_v2_run')\nEPISODIC=BRAIN_V2/'episodic_memory.jsonl'\nMEMORY_DIR=Path('/content/drive/MyDrive/EvolveAI/brain_v3_memory')\nif not EPISODIC.is_file(): raise FileNotFoundError(f'Brain v2 episodic log not found: {EPISODIC}')\nMEMORY_DIR.mkdir(parents=True,exist_ok=True)\nprint('Source:',EPISODIC); print('Destination:',MEMORY_DIR)\n

In [ ]:
from brain.semantic_memory import SemanticMemory\nmemory=SemanticMemory(MEMORY_DIR)\ningestion=memory.ingest_episodic_jsonl(EPISODIC)\nprint('INGESTION REPORT')\nfor k,v in ingestion.items(): print(f'{k}: {v}')\n

In [ ]:
queries=['mathematics problem solution','language stories','news report','encyclopedic knowledge']\nfor query in queries:\n    print('\\nQUERY:',query)\n    for hit in memory.search(query,top_k=3):\n        print('score:',hit['score'],'domain:',hit['domain'],'sources:',hit['source_ids'])\n        print(hit['text'][:300]); print('concepts:',hit['concepts'][:8])\n

In [ ]:
print('TOP CONCEPT CANDIDATES')\nranked=sorted(memory.nodes.items(),key=lambda x:x[1]['document_count'],reverse=True)[:30]\nfor term,node in ranked: print(term,node['document_count'],node['domains'])\nif ranked:\n    term=ranked[0][0]; print('CO-OCCURRENCE LINKS:',term)\n    for item in memory.related_concepts(term,15): print(item)\nprint('MANIFEST:',(MEMORY_DIR/'semantic_memory_manifest.json').read_text())\n

## Limitations\n\n- Episodic memory stores only up to two sample texts per domain per batch, not every raw training record.\n- Candidate concepts are frequency-ranked words and adjacent bigrams, not a validated ontology.\n- Graph edges indicate co-occurrence in a source record, not causality or entailment.\n- Retrieval is lexical BM25-style search; it is not embedding-based semantic search and does not generate answers.\n- This build does not change SNN weights or prove autonomous reasoning.